# Engine validation

Runs the complete offline engine chain (schema, lint, expander, IPAM, compiler) over the manual example set and reports failures.

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
!python -m pytest tests/autopilot/test_s5_intent_schema.py -q
!python -m pytest tests -q -x --ignore=tests/autopilot/test_live_preflight.py